# Si-Ge cluster expansion workflow - part 1

This is a CASM project tutorial to generate a phase diagram using a Si-Ge binary alloy cluster expansion fit to DFT calculations. The overall workflow is split into two parts.

Topics covered in part 1:

1. **Project initialization**: Define the primitive crystal structure and allowed atoms on each crystal site
2. **Enumeration**: Enumerate crystal structures which are symmetrically distinct orderings of the atoms allowed by the prim occupation DoF
3. **Calculation**: Calculate the energies of the enumerated structures using DFT
4. **Import and mapping**: Import calculation results, mapping to orderings on the prim
5. **Set reference states**: Choose reference states to define a formation energy for each structure
6. **Query**: Query calculation properties


In [1]:
import pathlib

import libcasm.xtal as xtal
from casm.project import Project
from casm.tools.shared.json_io import safe_dump

input_dir = pathlib.Path("input")

project_path = pathlib.Path("SiGe_occ")
project_path.mkdir(parents=True, exist_ok=True)

## Project initialization

### Specify the "prim"

A primitive crystal structure and allowed degrees of freedom (the "prim") specifies:

- lattice vectors
- crystal basis sites
- global degrees of freedom
- site degrees of freedom, including allowed occupant species on each basis site.

When combined with a choice of basis function type, order, and truncation, the prim provides all the information needed to generate cluster expansion basis functions.

Here is the prim for the Si-Ge binary alloy project, which we write to a JSON-formatted file named *prim.json*:

In [2]:
prim_data = {
    "title": "SiGe_occ",
    "lattice_vectors": [
        [0.000000000000, 2.800000000000, 2.800000000000],  # 1st lattice vector
        [2.800000000000, 0.000000000000, 2.800000000000],  # 2nd lattice vector
        [2.800000000000, 2.800000000000, 0.000000000000],  # 3rd lattice vector
    ],
    "coordinate_mode": "Fractional",
    "basis": [
        {
            "coordinate": [0.0, 0.0, 0.0],
            "occupant_dof": ["Si", "Ge"],
        },
        {
            "coordinate": [0.25, 0.25, 0.25],
            "occupant_dof": ["Si", "Ge"],
        },
    ],
}

with open(project_path / "prim.json", "w") as f:
    f.write(xtal.pretty_json(prim_data))

For this particular project, the prim contains:

- **lattice_vectors**: A list of crystal lattice vectors. Units are typically Angstrom, but are ultimately determined by the method used to perform calculations. 
- **basis**: A list of crystal basis sites, including coordinate and allowed degrees of freedom. For this ZrO project, the basis sites contain:
  - **coordinate**: The location of the basis site, according to the "coordinate_mode".
  - **occupants**: A list of the possible occupant species that may reside at each site. The names are case sensitive, and “Va” is reserved for vacancies.
- **coordinate_mode**: Defines the units of basis site coordinates. May be one of:
  - "Cartesian": To specify basis coordinates using Cartesian coordinates:
    $$ r_{cart} = (x, y, z) $$
  - "Fractional" or "Direct": To specify basis coordinates defined in terms of the lattice vectors:
    $$ r_{cart} = L r_{frac}, $$
    where:
    - $r_{frac}$ are the coordinates in the fractional representation
    - $r_{cart}$ are the coordinates in the Cartesian representation
    - $L$ is the lattice as a column-vector matrix. 
  
**Note**: It is common, but not required, to use the results of a fully relaxed calculation of the structure with the default occupation values for the prim lattice vectors. The default occupation on each site is the species listed first in "occupants". For occupation cluster expansions, ideal supercells of the prim lattice are used for the initial state of DFT calculations and are the default reference for strain.

### Initialize a CASM project

A CASM project is a directory containing data related to a particular prim. The CASM project directory structure standardizes the location of various files used by multiple CASM methods. This makes it easier to perform the most common operations and easier to share a project with others.

A CASM project is initialized by defining a prim and using [Project.init TODO](TODO). This will:

1. Check if the prim has a primitive unit cell with a CASM standard lattice orientation
2. Perform a symmetry analysis
3. Generate some default directories, data, and settings
4. Perform a configuration check 


Notes:

- Project files that the user should not typically modify directly, including a copy of the prim, are stored in a hidden *.casm* sub-directory of the CASM project directory. The presense or absence of the *.casm* directory is used by CASM to detect a CASM project.


In [3]:
project = Project.init(path=project_path)

CASM project already exists at SiGe_occ
Using existing project


### Check prim symmetry

- It is good practice to confirm the prim has the expected symmetry.
- This helps to catch errors in the prim, or a lack of necessary precision in the lattice vectors or basis coordinates.

In [4]:
project.sym.print_factor_group()

0: 1
1: 4⁺ (-0.2500000  0.2500000  0.2500000) 0.25+x, 0.25-x, -0.25-x
2: 4⁺ ( 0.2500000 -0.2500000  0.2500000) -0.25+x, 0.25-x, 0.25+x
3: 4⁺ ( 0.2500000  0.2500000 -0.2500000) 0.25+x, -0.25+x, 0.25-x
4: 4⁻ (-0.2500000  0.2500000  0.2500000) 0.25+x, -0.25-x, 0.25-x
5: 4⁻ ( 0.2500000 -0.2500000  0.2500000) 0.25+x, 0.25-x, -0.25+x
6: 4⁻ ( 0.2500000  0.2500000 -0.2500000) -0.25+x, 0.25+x, 0.25-x
7: 3⁺ 3*x, -x, -x
8: 3⁺ x, x, -3*x
9: 3⁺ x, -3*x, x
10: 3⁺ x, x, x
11: 3⁻ 3*x, -x, -x
12: 3⁻ x, x, -3*x
13: 3⁻ x, -3*x, x
14: 3⁻ x, x, x
15: 2 0.125+x, 0.125, 0.125-x
16: 2 0.125+x, 0.125-x, 0.125
17: 2 0.125, 0.125+y, 0.125-y
18: 2 (-0.0000000  0.0000000  0.5000000) 0.125, 0.125, z
19: 2 (0.0000000 0.5000000 0.0000000) 0.125, y, 0.125
20: 2 ( 0.5000000 -0.0000000  0.0000000) x, 0.125, 0.125
21: 2 x, -x, -x
22: 2 x, -x, x
23: 2 x, x, -x
24: m x, y, x
25: m x, x, z
26: m x, y, y
27: m 2*x, 2*y, -x-y
28: m 2*x, -x+y, -2*y
29: m x, y, -2*x-y
30: g ( 0.5000000 -0.0000000  0.0000000) -0.125+x, 0.125+y, 

## Enumeration

### Introduction

To fit a cluster expansion for the Si-Ge system, we need a set of calculated energies for Si-Ge crystal structures with various orderings to use as training data. To begin, we use CASM to enumerate symmetrically distinct [Supercell](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.Supercell.html#supercell) and [Configuration](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.Configuration.html#configuration):

- A [Supercell](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.Supercell.html#supercell) defines the three-dimensional translations that repeat a crystal structure. 

  -  A supercell can be specified by the integer transformation matrix, $T$, relating the superstructure lattice vectors, $S$, to the unit structure lattice vectors, $L$, according to $S = L T$, where $S$ and $L$ are shape=(3,3) matrices with lattice vectors as columns.
 
- A [Configuration](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.Configuration.html#configuration) is a compact representation of the unit cell for a crystal structure that is allowed by the DoF specified in the prim. For this Si-Ge project, a configuration can be specified by:
 
  - the supercell that is the unit cell for the crystal structure, and
  - an integer occupation array (i.e. [1, 0, 1, ...]), indicating if Si or Ge is on each site in the supercell.

The mapping of occupation index to atom type is determined by the order occupants are listed in the Prim (0="Si", 1="Ge"). The order of indices is based on the location of the site in the [Supercell](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.Supercell.html#supercell), which has methods that allow converting between the site coordinates and the index of the site in the occupation array (*linear_site_index*).

The Supercell object holds symmetry representations that efficiently apply symmetry operations to a configuration, permuting the occupation indices. This allows CASM to efficiently perform comparisons and check if configurations are symmetrically distinct. The same symmetry representations can be used to transform any configuration in the same supercell, so a Supercell object can be shared by multiple Configuration objects.

### Supercell enumeration

#### Enumerating supercells by volume

To start, we construct a new enumeration with id="supercells_by_volume.1" using [project.enum.get](), which returns an [EnumData]() object. 

The method [EnumData.supercells_by_volume]() enumerates symmetrically distinct supercells from a minimum to a maximum volume, specified as integer multiples of the prim unit cell volume. It also has additional options, described in the reference documentation, for more complex use cases:

- Enumerate supercells of another supercell
- Enumerate 1d or 2d supercells
- Enumerate supercells with a fixed shape but different sizes

In [5]:
# Enumerate supercells with volume 1 to 4
enum = project.enum.get("supercells_by_volume.1")
enum.supercells_by_volume(
    max=4,
    min=1,
    verbose=True,
)

-- Begin: Enumerating supercells by volume --

  Generated: SCEL1_1_1_1_0_0_0 (already existed)
  Generated: SCEL2_2_1_1_0_1_1 (already existed)
  Generated: SCEL2_2_1_1_0_0_1 (already existed)
  Generated: SCEL3_3_1_1_0_2_2 (already existed)
  Generated: SCEL3_3_1_1_0_2_1 (already existed)
  Generated: SCEL3_3_1_1_0_0_2 (already existed)
  Generated: SCEL4_4_1_1_0_0_0 (already existed)
  Generated: SCEL4_4_1_1_0_1_0 (already existed)
  Generated: SCEL4_4_1_1_0_0_2 (already existed)
  Generated: SCEL4_4_1_1_0_0_3 (already existed)
  Generated: SCEL4_4_1_1_0_2_1 (already existed)
  Generated: SCEL4_2_2_1_0_1_0 (already existed)
  Generated: SCEL4_2_2_1_1_1_0 (already existed)
  DONE

-- Summary --

  Initial number of supercells: 13
  Final number of supercells: 13
  Enumerated 13 supercells (0 new, 13 existing).

overwrite: SiGe_occ/enumerations/enum.supercells_by_volume.1/meta.json
overwrite: SiGe_occ/enumerations/enum.supercells_by_volume.1/scel_set.json


#### Enumeration Data

The results of an enumeration can be accessed using the [EnumData]() class. We can put additional information, including a text description of the enumeration, in the [EnumData.meta]() dict and save the updated EnumData using [commit](). Subsequently, if "desc" exists in [EnumData.meta](), it will be printed along with summary information such as the number of supercells.

In [6]:
enum.meta = {"desc": "Initial supercell enumeration"}
enum.commit()
print(enum)

overwrite: SiGe_occ/enumerations/enum.supercells_by_volume.1/meta.json
overwrite: SiGe_occ/enumerations/enum.supercells_by_volume.1/scel_set.json
EnumData:
- id: supercells_by_volume.1
- desc: "Initial supercell enumeration"
- supercell_set: 13 supercells


Later, the [EnumData]() may also be accessed by id string using the [enum.get]() method.

In [7]:
enum = project.enum.get("supercells_by_volume.1")
print(enum)

EnumData:
- id: supercells_by_volume.1
- desc: "Initial supercell enumeration"
- supercell_set: 13 supercells


#### SupercellSet

The supercells enumerated by [enum.supercells_by_volume]() are stored as [SupercellRecord](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.SupercellRecord.html#supercellrecord) in a [SupercellSet](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.SupercellSet.html#supercellset). Each SupercellRecord includes a [Supercell](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.Supercell.html#supercell) and some additional information about the supercell, including a [supercell_name]() string which is used as an identifier.

A SupercellSet:

- does not keep multiple SupercellRecord for supercells that have the same superlattice vectors;
- does allow storing separate SupercellRecord for supercells which are distinct (have different superlattice vectors) but are symmetrically equivalent (superlattice points are mapped by a crystal group operation).

Iterating over a [SupercellSet](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.SupercellSet.html#supercellset) yields [SupercellRecord](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.SupercellRecord.html#supercellrecord). Each SupercellRecord includes a [Supercell](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.Supercell.html#supercell) and some additional information about the Supercell.

In [8]:
# Iterate over the first three SupercellRecord
# in the SupercellSet and print the record
for i, record in enumerate(enum.supercell_set):
    print(record)
    if i == 2:
        break

{
  "canonical_supercell_name": "SCEL1_1_1_1_0_0_0",
  "is_canonical": true,
  "supercell": {
    "supercell_name": "SCEL1_1_1_1_0_0_0",
    "transformation_matrix_to_supercell": [
      [1, 0, 0],
      [0, 1, 0],
      [0, 0, 1]
    ]
  },
  "supercell_name": "SCEL1_1_1_1_0_0_0"
}
{
  "canonical_supercell_name": "SCEL2_2_1_1_0_1_1",
  "is_canonical": true,
  "supercell": {
    "supercell_name": "SCEL2_2_1_1_0_1_1",
    "transformation_matrix_to_supercell": [
      [1, 0, 1],
      [0, 1, 1],
      [-1, -1, 0]
    ]
  },
  "supercell_name": "SCEL2_2_1_1_0_1_1"
}
{
  "canonical_supercell_name": "SCEL2_2_1_1_0_0_1",
  "is_canonical": true,
  "supercell": {
    "supercell_name": "SCEL2_2_1_1_0_0_1",
    "transformation_matrix_to_supercell": [
      [0, -1, -1],
      [0, 1, -1],
      [1, 0, 1]
    ]
  },
  "supercell_name": "SCEL2_2_1_1_0_0_1"
}


#### Storing multiple enumerations

Enumerations are stored in directories based on their id string. If an id is not given, or has value None, a new enumeration is automatically generated in sequential order. If the id of an existing enumeration is given, that enumeration is updated with any additional supercells generated.


In [9]:
# Enumerate supercells with volume 3 to 5
enum = project.enum.get("supercells_by_volume.2")
enum.supercells_by_volume(max=5, min=3)
print()
print(enum)

-- Begin: Enumerating supercells by volume --

  Generated: SCEL3_3_1_1_0_2_2 (already existed)
  Generated: SCEL3_3_1_1_0_2_1 (already existed)
  Generated: SCEL3_3_1_1_0_0_2 (already existed)
  Generated: SCEL4_4_1_1_0_0_0 (already existed)
  Generated: SCEL4_4_1_1_0_1_0 (already existed)
  Generated: SCEL4_4_1_1_0_0_2 (already existed)
  Generated: SCEL4_4_1_1_0_0_3 (already existed)
  Generated: SCEL4_4_1_1_0_2_1 (already existed)
  Generated: SCEL4_2_2_1_0_1_0 (already existed)
  Generated: SCEL4_2_2_1_1_1_0 (already existed)
  Generated: SCEL5_1_1_5_0_0_0 (already existed)
  Generated: SCEL5_5_1_1_0_4_3 (already existed)
  Generated: SCEL5_5_1_1_0_0_3 (already existed)
  Generated: SCEL5_5_1_1_0_0_4 (already existed)
  Generated: SCEL5_5_1_1_0_1_3 (already existed)
  DONE

-- Summary --

  Initial number of supercells: 15
  Final number of supercells: 15
  Enumerated 15 supercells (0 new, 15 existing).

overwrite: SiGe_occ/enumerations/enum.supercells_by_volume.2/scel_set.json

E

### Configuration enumeration

#### Enumerating configurations by supercell

The method [occ_by_supercell](TODO) enumerates all occupations in supercells ranging from a minimum to a maximum volume. All configurations are guaranteed to be in a canonical supercell. By default it:

- only outputs primitive configurations,
- only outputs configurations in canonical form (the configuration that compares greatest to all configurations in a supercell that can be mapped by symmetry operations).

With these defaults, if enumeration proceeds without skipping supercells, all symmetrically distinct configurations will be enumerated.

As with enum.supercells_by_volume it also has additional options, described in the reference documentation, for more complex use cases:

- Enumerate occupations in supercells of another supercell
- Enumerate occupations in 1d or 2d supercells
- Enumerate occupations in supercells with a fixed shape but different sizes

**Warning**: The number of possible occupations in a $n$-component alloy with $m$ sites is $n^m$. Take care not to request too large of an enumeration. 
    

In [10]:
# Enumerate configurations in supercells with volume 1 to 4
enum = project.enum.get("occ_by_supercell.1")
enum.occ_by_supercell(max=4, min=1)

-- Begin: Enumerating occupations by supercell --

Enumerate configurations for: SCEL1_1_1_1_0_0_0
3 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL2_2_1_1_0_1_1
4 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL2_2_1_1_0_0_1
3 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL3_3_1_1_0_2_2
13 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL3_3_1_1_0_2_1
10 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL3_3_1_1_0_0_2
10 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL4_4_1_1_0_0_0
36 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL4_4_1_1_0_1_0
27 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL4_4_1_1_0_0_2
36 configurations (0 new, 0 excluded by filter)

Enumerate configurations for: SCEL4_4_1_1_0_0_3
24 configurations (0 new, 0 exc

#### ConfigurationSet

The configurations enumerated by [enum.occ_by_supercell]() are stored as [ConfigurationRecord](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.ConfigurationRecord.html#configurationrecord) in a [ConfigurationSet](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.ConfigurationSet.html#configurationset). Each ConfigurationRecord includes a [Configuration](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.Configuration.html#configuration) and some additional information about the configuration, including a [configuration_name]() string which is used as an identifier.

A ConfigurationSet:

- requires configuration be in a canonical supercell;
- does not keep multiple ConfigurationRecord for configurations that have the same DoF values;
- does allow storing separate ConfigurationRecord for configurations which are distinct (have different DoF values) but are symmetrically equivalent (DoF values are mapped by a symmetry operation).
- users are responsible for placing any other constraints (canonical configurations only, primitive configurations only, etc.) on which configuration are added to ConfigurationSet.

<div style="background-color: #ffebb9; padding: 10px; margin: 10px;">
    <b style="color: #B27000;">Note on using ConfigurationSet:</b>
    <p>ConfigurationSet is optimized for finding unique configurations using canonical supercells. Users <em>must ensure that configuration added to ConfigurationSet are in a canonical supercell</em>. This is not checked by ConfigurationSet but required to ensure proper configuration naming, serialization, and deserialization. Configurations that are not in a canonical supercell should be stored in a list or some other data structure.</p>
</div>

Iterating over a [ConfigurationSet](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.ConfigurationSet.html#configurationset) yields [ConfigurationRecord](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.ConfigurationRecord.html#configurationrecord).


In [11]:
# Iterate over the first three ConfigurationRecord
# in the ConfigurationSet and print the record
for i, record in enumerate(enum.configuration_set):
    print(record)
    if i == 3:
        break

{
  "configuration": {
    "basis": "standard",
    "dof": {
      "occ": [0, 0]
    },
    "supercell_name": "SCEL1_1_1_1_0_0_0",
    "transformation_matrix_to_supercell": [
      [1, 0, 0],
      [0, 1, 0],
      [0, 0, 1]
    ]
  },
  "configuration_id": "0",
  "configuration_name": "SCEL1_1_1_1_0_0_0/0",
  "supercell_name": "SCEL1_1_1_1_0_0_0"
}
{
  "configuration": {
    "basis": "standard",
    "dof": {
      "occ": [1, 0]
    },
    "supercell_name": "SCEL1_1_1_1_0_0_0",
    "transformation_matrix_to_supercell": [
      [1, 0, 0],
      [0, 1, 0],
      [0, 0, 1]
    ]
  },
  "configuration_id": "1",
  "configuration_name": "SCEL1_1_1_1_0_0_0/1",
  "supercell_name": "SCEL1_1_1_1_0_0_0"
}
{
  "configuration": {
    "basis": "standard",
    "dof": {
      "occ": [1, 1]
    },
    "supercell_name": "SCEL1_1_1_1_0_0_0",
    "transformation_matrix_to_supercell": [
      [1, 0, 0],
      [0, 1, 0],
      [0, 0, 1]
    ]
  },
  "configuration_id": "2",
  "configuration_name": "SCEL1_1_

#### Conversion to structure

The [Configuration.to_structure](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.Configuration.to_structure.html#libcasm.configuration.Configuration.to_structure) method converts a CASM [Configuration](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.Configuration.html#libcasm.configuration.Configuration) to a CASM [Structure](https://prisms-center.github.io/CASMcode_pydocs/libcasm/xtal/2.0/reference/libcasm/_autosummary/libcasm.xtal.Structure.html#libcasm.xtal.Structure). A Structure:

- represents a crystal structure with a 3d lattice,
- is not restricted to the DoF values allowed by a prim,
- has built in methods for conversions to and from VASP POSCAR format.

In [12]:
# Iterate over the first three ConfigurationRecord
# in the ConfigurationSet and print the record
for i, record in enumerate(enum.configuration_set):
    name = record.configuration_name
    structure = record.configuration.to_structure()
    poscar_str = structure.to_poscar_str(title=name)

    print("~~~")
    print(f"Configuration: {name}")
    print(f"Structure: {structure}")
    print(f"POSCAR:\n{poscar_str}", end="")
    if i == 2:
        break

~~~
Configuration: SCEL1_1_1_1_0_0_0/0
Structure: {
  "atom_coords": [
    [0.0, 0.0, 0.0],
    [0.25000000000000006, 0.25, 0.25000000000000006]
  ],
  "atom_type": ["Si", "Si"],
  "coordinate_mode": "Fractional",
  "lattice_vectors": [
    [0.0, 2.8, 2.8],
    [2.8, 0.0, 2.8],
    [2.8, 2.8, 0.0]
  ]
}
POSCAR:
SCEL1_1_1_1_0_0_0/0
1.00000000
0.00000000 2.80000000 2.80000000
2.80000000 0.00000000 2.80000000
2.80000000 2.80000000 0.00000000
Si 
2 
Direct
0.00000000 0.00000000 0.00000000 Si
0.25000000 0.25000000 0.25000000 Si

~~~
Configuration: SCEL1_1_1_1_0_0_0/1
Structure: {
  "atom_coords": [
    [0.0, 0.0, 0.0],
    [0.25000000000000006, 0.25, 0.25000000000000006]
  ],
  "atom_type": ["Ge", "Si"],
  "coordinate_mode": "Fractional",
  "lattice_vectors": [
    [0.0, 2.8, 2.8],
    [2.8, 0.0, 2.8],
    [2.8, 2.8, 0.0]
  ]
}
POSCAR:
SCEL1_1_1_1_0_0_0/1
1.00000000
0.00000000 2.80000000 2.80000000
2.80000000 0.00000000 2.80000000
2.80000000 2.80000000 0.00000000
Ge Si 
1 1 
Direct
0.000000

#### Order in ConfigurationSet

ConfigurationSet assigns indices used in configuration names ("\<supercell_name\>/\<index\>") sequentially, per supercell, as the configurations are added, but sorts configurations by the degree of freedom values (i.e. occupation variables, strain variables, etc.).

Therefore, iterating over configurations and printing their names may make it appear that they are "mixed up", but this is by design.

In [13]:
# Iterate over the first 20 ConfigurationRecord in the ConfigurationSet,
# and print the configuration name and occupation indices
print(f'{"ConfigurationName":24}{"OccupationIndices":20}{"AtomType":40}')
print("-" * 84)
for i, record in enumerate(enum.configuration_set):
    name = record.configuration_name
    occupation = record.configuration.occupation
    structure = record.configuration.to_structure()
    atom_type = structure.atom_type()
    print(f"{name:24}{str(occupation):20}{str(atom_type):40}")
    if i == 20:
        break

ConfigurationName       OccupationIndices   AtomType                                
------------------------------------------------------------------------------------
SCEL1_1_1_1_0_0_0/0     [0 0]               ['Si', 'Si']                            
SCEL1_1_1_1_0_0_0/1     [1 0]               ['Ge', 'Si']                            
SCEL1_1_1_1_0_0_0/2     [1 1]               ['Ge', 'Ge']                            
SCEL2_2_1_1_0_1_1/0     [1 0 0 0]           ['Ge', 'Si', 'Si', 'Si']                
SCEL2_2_1_1_0_1_1/3     [1 0 0 1]           ['Ge', 'Si', 'Si', 'Ge']                
SCEL2_2_1_1_0_1_1/1     [1 0 1 0]           ['Ge', 'Si', 'Ge', 'Si']                
SCEL2_2_1_1_0_1_1/2     [1 1 1 0]           ['Ge', 'Ge', 'Ge', 'Si']                
SCEL2_2_1_1_0_0_1/0     [1 0 0 0]           ['Ge', 'Si', 'Si', 'Si']                
SCEL2_2_1_1_0_0_1/1     [1 0 1 0]           ['Ge', 'Si', 'Ge', 'Si']                
SCEL2_2_1_1_0_0_1/2     [1 1 1 0]           ['Ge', 'Ge', 'Ge', 'S

#### Configuration list

Configurations can also be stored in a list, [enum.configuration_list](https://prisms-center.github.io/CASMcode_pydocs/casm/project/2.0/reference/casm/_autosummary/casm.project.enum.EnumData.configuration_list.html#casm.project.enum.EnumData.configuration_list). It is up to the user to enforce any conditions on configuration stored in the list, such as whether the list is sorted or only contains canonical configurations. 

After changes to the enumeration's [configuration_set](https://prisms-center.github.io/CASMcode_pydocs/casm/project/2.0/reference/casm/_autosummary/casm.project.enum.EnumData.configuration_set.html#casm.project.enum.EnumData.configuration_set) or [configuration_list](https://prisms-center.github.io/CASMcode_pydocs/casm/project/2.0/reference/casm/_autosummary/casm.project.enum.EnumData.configuration_list.html#casm.project.enum.EnumData.configuration_list) attributes, the [commit](https://prisms-center.github.io/CASMcode_pydocs/casm/project/2.0/reference/casm/_autosummary/casm.project.enum.EnumData.commit.html#casm.project.enum.EnumData.commit) method can be called to save the changes. If the files are changed independently, then the [load](https://prisms-center.github.io/CASMcode_pydocs/casm/project/2.0/reference/casm/_autosummary/casm.project.enum.EnumData.load.html#casm.project.enum.EnumData.load) method can be used to read the files. 

Here we:
- select a configuration,
- generate all equivalent orderings that fit in the same supercell,
- add distinct configurations to the configuration list, without duplication, and
- save the configurations.

In [14]:
import libcasm.configuration as casmconfig
from casm.tools.shared.json_io import read_required

print("Initial enum.configuration_list size:", len(enum.configuration_list))

# Get a configuration by name
config = enum.configuration_set.get("SCEL4_2_2_1_1_1_0/4").configuration
print("Site occupant indices:")
print(config.occupation)
print()

# Generate distinct equivalents by applying symmetry operations that
# leave the supercell lattice unchanged but permute site occupants
equiv = casmconfig.make_equivalent_configurations(config)

# Add the equivalent configurations, with a check to avoid exact duplicates
for x in equiv:
    if x not in enum.configuration_list:
        enum.configuration_list.append(x)

print("Final enum.configuration_list size:", len(enum.configuration_list))
print("Equivalent site occupant indices:")
for i, config in enumerate(enum.configuration_list):
    print(f"{i}:", config.occupation)
print()

Initial enum.configuration_list size: 0
Site occupant indices:
[1 1 1 0 1 0 0 0]

Final enum.configuration_list size: 32
Equivalent site occupant indices:
0: [0 0 0 1 0 1 1 1]
1: [0 0 0 1 1 0 1 1]
2: [0 0 0 1 1 1 0 1]
3: [0 0 0 1 1 1 1 0]
4: [0 0 1 0 0 1 1 1]
5: [0 0 1 0 1 0 1 1]
6: [0 0 1 0 1 1 0 1]
7: [0 0 1 0 1 1 1 0]
8: [0 1 0 0 0 1 1 1]
9: [0 1 0 0 1 0 1 1]
10: [0 1 0 0 1 1 0 1]
11: [0 1 0 0 1 1 1 0]
12: [0 1 1 1 0 0 0 1]
13: [0 1 1 1 0 0 1 0]
14: [0 1 1 1 0 1 0 0]
15: [0 1 1 1 1 0 0 0]
16: [1 0 0 0 0 1 1 1]
17: [1 0 0 0 1 0 1 1]
18: [1 0 0 0 1 1 0 1]
19: [1 0 0 0 1 1 1 0]
20: [1 0 1 1 0 0 0 1]
21: [1 0 1 1 0 0 1 0]
22: [1 0 1 1 0 1 0 0]
23: [1 0 1 1 1 0 0 0]
24: [1 1 0 1 0 0 0 1]
25: [1 1 0 1 0 0 1 0]
26: [1 1 0 1 0 1 0 0]
27: [1 1 0 1 1 0 0 0]
28: [1 1 1 0 0 0 0 1]
29: [1 1 1 0 0 0 1 0]
30: [1 1 1 0 0 1 0 0]
31: [1 1 1 0 1 0 0 0]



In [15]:
# Commit changes:
enum.commit()

overwrite: SiGe_occ/enumerations/enum.occ_by_supercell.1/scel_set.json
overwrite: SiGe_occ/enumerations/enum.occ_by_supercell.1/config_set.json
write: SiGe_occ/enumerations/enum.occ_by_supercell.1/config_list.json


In [16]:
# Show config_list.json:
data = read_required(enum.enum_dir / "config_list.json")
print(xtal.pretty_json(data))
print()

[
  {
    "basis": "standard",
    "dof": {
      "occ": [0, 0, 0, 1, 0, 1, 1, 1]
    },
    "supercell_name": "SCEL4_2_2_1_1_1_0",
    "transformation_matrix_to_supercell": [
      [-1, 1, 1],
      [1, -1, 1],
      [1, 1, -1]
    ]
  },
  {
    "basis": "standard",
    "dof": {
      "occ": [0, 0, 0, 1, 1, 0, 1, 1]
    },
    "supercell_name": "SCEL4_2_2_1_1_1_0",
    "transformation_matrix_to_supercell": [
      [-1, 1, 1],
      [1, -1, 1],
      [1, 1, -1]
    ]
  },
  {
    "basis": "standard",
    "dof": {
      "occ": [0, 0, 0, 1, 1, 1, 0, 1]
    },
    "supercell_name": "SCEL4_2_2_1_1_1_0",
    "transformation_matrix_to_supercell": [
      [-1, 1, 1],
      [1, -1, 1],
      [1, 1, -1]
    ]
  },
  {
    "basis": "standard",
    "dof": {
      "occ": [0, 0, 0, 1, 1, 1, 1, 0]
    },
    "supercell_name": "SCEL4_2_2_1_1_1_0",
    "transformation_matrix_to_supercell": [
      [-1, 1, 1],
      [1, -1, 1],
      [1, 1, -1]
    ]
  },
  {
    "basis": "standard",
    "dof": {
    

#### Selecting configurations

A [ConfigSelection](TODO) allows iterating over configurations stored in an enumeration's [configuration_set](https://prisms-center.github.io/CASMcode_pydocs/casm/project/2.0/reference/casm/_autosummary/casm.project.enum.EnumData.configuration_set.html#casm.project.enum.EnumData.configuration_set) and [configuration_list](https://prisms-center.github.io/CASMcode_pydocs/casm/project/2.0/reference/casm/_autosummary/casm.project.enum.EnumData.configuration_list.html#casm.project.enum.EnumData.configuration_list) in a single pass. 

**Iterating:**

A ConfigSelection stores whether configurations are "selected" or "unselected" in a file in the enumeration directory. The selection of configurations can then be easily saved, loaded, updated, and used.

Iterating over a ConfigSelection yields a [ConfigSelectionRecord](TODO) which has properties such as [*name*](), [*configuration*](), [*is_selected*](), and [*n_unitcells*](). There are multiples options for iterating over the records:

- *for record in config_selection* - By default, yield for selected configurations only.
- *for record in config_selection.selected* - Also yields for selected configurations only, being explicit.
- *for record in config_selection.all* - Yield for all configurations, selected or unselected.
- *for record in config_selection.unselected* - Yield for only unselected configurations.


**Use cases:**

ConfigSelection also:

- handles accessing project data to make it easier to calculate configuration properties such as the parametric composition or correlations for a particular basis set, and
- helps setting up calculations, handling job submission, and collecting calculation results.


**Example:**

By default, a new ConfigSelection includes all configurations in its parent enumeration as "selected". Here we:

- construct a new ConfigSelection named "main",
- print selection summary information,
- iterate over records, selecting configurations and printing some information,
- save the selection, and
- re-load the selection from file.


In [17]:
# Construct a ConfigSelection
# - Load from file if it already exists
# - Otherwise, create a new ConfigSelection with all configurations selected
config_selection = enum.config_selection(name="main")

# Print summary information
print(config_selection)
print()

# Select only configurations with n_unitcells=2,
# and print site occupant indices
for record in config_selection.all:
    if record.n_unitcells == 2:
        record.select()
        print(record.name, ":", record.configuration.occupation)
    else:
        record.deselect()

print("Number selected:", config_selection.n_selected)
print()

# Save the selection in the enumeration directory
# to a file named `config_selection.main.json`
print("Saving 'main'...")
config_selection.commit()
print()

# Load the selection from file
print("Loading 'main'...")
new_config_selection = enum.config_selection(name="main")
print("Number selected:", config_selection.n_selected)
print()

ConfigSelection:
- name: main
- enum: occ_by_supercell.1
- clex: vasp-parameter-set-1
  - calctype: vasp-parameter-set-1
  - ref: default
  - bset: default
  - eci: default
- n_total: 214
- n_selected: 3
- n_unselected: 211

SCEL2_2_1_1_0_1_1/0 : [1 0 0 0]
SCEL2_2_1_1_0_1_1/3 : [1 0 0 1]
SCEL2_2_1_1_0_1_1/1 : [1 0 1 0]
SCEL2_2_1_1_0_1_1/2 : [1 1 1 0]
SCEL2_2_1_1_0_0_1/0 : [1 0 0 0]
SCEL2_2_1_1_0_0_1/1 : [1 0 1 0]
SCEL2_2_1_1_0_0_1/2 : [1 1 1 0]
Number selected: 7

Saving 'main'...
overwrite: SiGe_occ/enumerations/enum.occ_by_supercell.1/config_selection.main.json

Loading 'main'...
Number selected: 7



#### Clearing configurations

For fitting the Si-Ge cluster expansion we don't need the equivalent configurations stored in *configuration_list*. So we will:

- clear the *configuration_list*,
- and remove the "main" selection.

<div style="background-color: #ffebb9; padding: 10px; margin: 10px;">
    <b style="color: #B27000;">Note on removing configurations:</b>
    <p>When the <em>configuration_list</em> is cleared and then new configurations are added, the configuration name (i.e. "config_list/&lt;index>") gets re-used and there is a potential for confusion. Clearing configurations from <em>configuration_list</em> and committing only removes the entry in "config_list.json" and does not clear anything else. Updating calculation directories, selections, etc. is left to the user to manage.</p> 
    <p>ConfigurationSet assigns indices used in configuration names ("&lt;supercell_name>/&lt;index>") sequentially, per supercell, as the configurations are added. These next index to use for each supercell is saved in the "config_set.json" file and indices are not re-used unless the "config_set.json" file is removed.</p>
</div>


In [18]:
# Clear the configuration_list
enum.configuration_list.clear()
enum.commit()

# Remove the "main" ConfigSelection
config_selection = enum.config_selection(name="main")
config_selection.remove()
del config_selection


overwrite: SiGe_occ/enumerations/enum.occ_by_supercell.1/scel_set.json
overwrite: SiGe_occ/enumerations/enum.occ_by_supercell.1/config_set.json
Removed selection file: SiGe_occ/enumerations/enum.occ_by_supercell.1/config_selection.main.json


#### Filtered enumeration

A custom filter function may be used to filter configurations during enumeration. Here we:

- use [ConfigCompositionCalculator]() to calculate the number of each type of atom in the supercell,
- keep configurations that have exactly 2 Ge,
- use ``dry_run=True``so the enumeration is not committed automatically.

In [19]:
# Enumerate configurations:
# - in supercells with volume 1 to 3
# - with exactly 2 Ge atoms in the supercell

from casm.project.enum import EnumData
from libcasm.configuration import Configuration, SupercellRecord

# Get the casm.project.ConfigCompositionCalculator
comp = project.make_chemical_comp_calculator()

# Get the index of Ge in the composition arrays
i_Ge = comp.components.index("Ge")

# Print each check?
verbose_checks = True


def filter_f(config: Configuration, enum: EnumData) -> bool:
    """Return True to include; False to exclude"""

    # Get number of Ge in the supercell
    N_Ge = comp.per_supercell(config)[i_Ge]

    # Print info about the config being checked
    if verbose_checks:
        record = SupercellRecord(config.supercell)
        print(
            f"~check~ {record.supercell_name}",
            config.occupation,
            f"include?: {N_Ge == 2}",
        )
    return N_Ge == 2


enum = project.enum.get("occ_by_supercell.2")
enum.occ_by_supercell(
    max=3,
    min=1,
    filter_f=filter_f,
    verbose=True,
    dry_run=True,
)

-- Begin: Enumerating occupations by supercell --

Enumerate configurations for: SCEL1_1_1_1_0_0_0
~check~ SCEL1_1_1_1_0_0_0 [0 0] include?: False
~check~ SCEL1_1_1_1_0_0_0 [1 0] include?: False
~check~ SCEL1_1_1_1_0_0_0 [1 1] include?: True
3 configurations (1 new, 2 excluded by filter)

Enumerate configurations for: SCEL2_2_1_1_0_1_1
~check~ SCEL2_2_1_1_0_1_1 [1 0 0 0] include?: False
~check~ SCEL2_2_1_1_0_1_1 [1 0 1 0] include?: True
~check~ SCEL2_2_1_1_0_1_1 [1 1 1 0] include?: False
~check~ SCEL2_2_1_1_0_1_1 [1 0 0 1] include?: True
4 configurations (2 new, 2 excluded by filter)

Enumerate configurations for: SCEL2_2_1_1_0_0_1
~check~ SCEL2_2_1_1_0_0_1 [1 0 0 0] include?: False
~check~ SCEL2_2_1_1_0_0_1 [1 0 1 0] include?: True
~check~ SCEL2_2_1_1_0_0_1 [1 1 1 0] include?: False
3 configurations (1 new, 2 excluded by filter)

Enumerate configurations for: SCEL3_3_1_1_0_2_2
~check~ SCEL3_3_1_1_0_2_2 [1 0 0 0 0 0] include?: False
~check~ SCEL3_3_1_1_0_2_2 [1 1 0 0 0 0] include?: Tru

#### Other enumeration methods

Internally, the [occ_by_supercell](TODO) method integrates the lower-level class [ConfigEnumAllOccupations](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.enumerate.ConfigEnumAllOccupations.html#libcasm.enumerate.ConfigEnumAllOccupations) from the [*libcasm.enumerate*](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.enumerate.html#module-libcasm.enumerate) package into *Project*. CASM includes other enumeration methods in *libcasm.enumerate* package that can be used directly. 

Other enumeration methods are not yet integrated into *Project*, but the methods in *libcasm.enumerate* can be used individually and the resulting configurations can be added to an enumeration's [configuration_set](https://prisms-center.github.io/CASMcode_pydocs/casm/project/2.0/reference/casm/_autosummary/casm.project.enum.EnumData.configuration_set.html#casm.project.enum.EnumData.configuration_set) and [configuration_list](https://prisms-center.github.io/CASMcode_pydocs/casm/project/2.0/reference/casm/_autosummary/casm.project.enum.EnumData.configuration_list.html#casm.project.enum.EnumData.configuration_list). 

Some of the other enumeration methods include:

- Occupation enumeration using [ConfigEnumAllOccupations](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.enumerate.ConfigEnumAllOccupations.html#libcasm.enumerate.ConfigEnumAllOccupations):
  - [by_cluster](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.enumerate.ConfigEnumAllOccupations.by_cluster.html#libcasm.enumerate.ConfigEnumAllOccupations.by_cluster): Enumerate occupation perturbations of a background configuration on specified clusters
  - [by_sublattice](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.enumerate.ConfigEnumAllOccupations.by_sublattice.html#libcasm.enumerate.ConfigEnumAllOccupations.by_sublattice): Enumerate occupation perturbations of a background configuration on specified sublattices
  - [by_integral_site_coordinates](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.enumerate.ConfigEnumAllOccupations.by_integral_site_coordinates.html#libcasm.enumerate.ConfigEnumAllOccupations.by_integral_site_coordinates): Enumerate occupation perturbations of a background configuration on specified sites
  - [by_supercell_with_continuous_dof](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.enumerate.ConfigEnumAllOccupations.by_supercell_with_continuous_dof.html#libcasm.enumerate.ConfigEnumAllOccupations.by_supercell_with_continuous_dof): Enumerate all occupations in a series of enumerated supercells, with non-default continuous DoF

- Occupation enumeration using [SuperConfigEnum](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.enumerate.SuperConfigEnum.html#libcasm.enumerate.SuperConfigEnum):
  - [by_supercell](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.enumerate.SuperConfigEnum.by_supercell.html#libcasm.enumerate.SuperConfigEnum.by_supercell): Make super configurations of the motif configuration, without changing the orientation of the motif
 
- Continuous degree of freedom (i.e. strain and displacement) enumeration using [ConfigEnumMeshGrid](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.enumerate.ConfigEnumMeshGrid.html#libcasm.enumerate.ConfigEnumMeshGrid):

  - [by_range](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.enumerate.ConfigEnumMeshGrid.by_range.html#libcasm.enumerate.ConfigEnumMeshGrid.by_range): Enumerate on a meshgrid, with coordinates specified by a range along each axis of a coordinate system in the continuous degree of freedom variable space
  - [by_grid_coordinates](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.enumerate.ConfigEnumMeshGrid.by_grid_coordinates.html#libcasm.enumerate.ConfigEnumMeshGrid.by_grid_coordinates): Enumerate on a meshgrid, specified with coordinates along each axis  of a coordinate system in the continuous degree of freedom variable space
  - [by_irreducible_wedge](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.enumerate.ConfigEnumMeshGrid.by_irreducible_wedge.html#libcasm.enumerate.ConfigEnumMeshGrid.by_irreducible_wedge): Enumerate on a meshgrid in each subwedge of the irreducible wedge of a coordinate system in the continuous degree of freedom variable space

- Occupation enumeration in the neighborhood of an atomic hop event (for kinetic Monte Carlo) using [ConfigEnumLocalOccupations](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.enumerate.ConfigEnumLocalOccupations.html#libcasm.enumerate.ConfigEnumLocalOccupations):

  - [by_cluster](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.enumerate.ConfigEnumLocalOccupations.by_cluster.html#libcasm.enumerate.ConfigEnumLocalOccupations.by_cluster): Enumerate occupation perturbations on local-clusters in the neighborhood of an event in a background configuration 

Over time, it is planned that more enumeration methods will be integrated for direct use via *Project*.

#### Note on configuration names

Configuration names are not (and are not meant to be) a globally unique or a one-to-one mapping with the configuration's occupation variables, etc. They are only meant to be unique within an enumeration. The following section goes over methods for checking equivalence of configurations.

#### Equivalence of configurations

Configurations are equivalent (not checking transformations by symmetry operations), if they have the same Prim, the same Supercell, and the same degrees of freedom (exactly equal occupation indices and approximately equal continuous variables).

To check if configurations are equal use:

```python
if config_A == config_B:
    # do something...
```

To check if an equivalent configuration is in a *ConfigurationSet* or *list[Configuration]* use:

```python
if config in set_or_list:
    # do something...
```

To find the name of an equivalent configuration in a *ConfigurationSet* use:

```python
# check if configuration is already in the ConfigurationSet:
configuration = Configuration(...)
record = configuration_set.get(configuration)
if record is not None:
    configuration_name = record.configuration_name
else:
    # configuration is not in the ConfigurationSet
    configuration_name = None
```

To find an equivalent configuration in a *list[Configuration]* use:
```python
# check if configuration is in a list:
configuration = Configuration(...)
try:
    index = configuration_list.index(configuration)
except ValueError as e:
    # configuration is not in the list
    index = None
```

**Check for equivalence under symmetry**

The above methods do not check if a symmetry operation maps one configuration onto another configuration (i.e. the same ordering, oriented differently and/or translated) in the same supercell.

To check for symmetry equivalence, first put the configurations in "canonical form" (the configuration that compares greatest to all configurations in a supercell that can be mapped by symmetry operations) and then compare them:

```python
from libcasm.configuration import (
    make_canonical_configuration,
)
canonical_A = make_canonical_configuration(config_A)
canonical_B = make_canonical_configuration(config_B)
if canonical_A == canonical_B:
    # do something...
```

Note that [make_canonical_configuration](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.make_canonical_configuration.html#make-canonical-configuration) can be expensive for large supercells. 

**Check for equivalence of configurations in different supercells**

Configurations in different supercells may represent the same ordering in the infinite crystal. For a complete check, they should be compared in canonical form in the same supercell. A general approach for checking if two configurations represent symmetrically equivalent orderings is:

```python
from libcasm.configuration import (
    make_canonical_configuration,
    make_primitive_configuration,
)

primitive_canonical_A = make_canonical_configuration(
    configuration=make_primitive_configuration(config_A),
    in_canonical_supercell=True,
)
primitive_canonical_B = make_canonical_configuration(
    configuration=make_primitive_configuration(config_B),
    in_canonical_supercell=True,
)
if primitive_canonical_A == primitive_canonical_B:
    # do something...
```



### Acting on enumeration data

This section provides a reference for various actions that can be performed on enumerations, and may be skipped for the Si-Ge demonstration project.

#### Get an enumeration by id

- Also, update enumeration metadata and commit.

In [20]:
enum = project.enum.get("supercells_by_volume.1")
enum.meta = {"desc": "Initial supercell enumeration"}
enum.commit()
print(enum)

overwrite: SiGe_occ/enumerations/enum.supercells_by_volume.1/meta.json
overwrite: SiGe_occ/enumerations/enum.supercells_by_volume.1/scel_set.json
EnumData:
- id: supercells_by_volume.1
- desc: "Initial supercell enumeration"
- supercell_set: 13 supercells


#### List all enumerations

- Print a summary of each enumeration in the project

In [21]:
project.enum.list()

EnumData:
- id: occ_by_supercell.1
- supercell_set: 13 supercells
- configuration_set: 214 configurations
EnumData:
- id: supercells_by_volume.1
- desc: "Initial supercell enumeration"
- supercell_set: 13 supercells
EnumData:
- id: supercells_by_volume.2
- supercell_set: 15 supercells


#### Copy an enumeration

- Will raise if the destination enumeration already exists

In [22]:
project.enum.copy(
    src_id="supercells_by_volume.2",
    dest_id="supercells_by_volume.3",
)
project.enum.list()

write: SiGe_occ/enumerations/enum.supercells_by_volume.3/scel_set.json
EnumData:
- id: occ_by_supercell.1
- supercell_set: 13 supercells
- configuration_set: 214 configurations
EnumData:
- id: supercells_by_volume.1
- desc: "Initial supercell enumeration"
- supercell_set: 13 supercells
EnumData:
- id: supercells_by_volume.2
- supercell_set: 15 supercells
EnumData:
- id: supercells_by_volume.3
- supercell_set: 15 supercells


#### Merge enumerations

- Supercells and configurations in source enumeration sets are inserted into the destination enumeration sets.
- Supercells and configurations in source enumeration lists are appended to the destination enumeration lists if they are not already present.

In [23]:
project.enum.merge(
    src_id="supercells_by_volume.1",
    dest_id="supercells_by_volume.3",
)
project.enum.list()

overwrite: SiGe_occ/enumerations/enum.supercells_by_volume.3/scel_set.json
EnumData:
- id: occ_by_supercell.1
- supercell_set: 13 supercells
- configuration_set: 214 configurations
EnumData:
- id: supercells_by_volume.1
- desc: "Initial supercell enumeration"
- supercell_set: 13 supercells
EnumData:
- id: supercells_by_volume.2
- supercell_set: 15 supercells
EnumData:
- id: supercells_by_volume.3
- supercell_set: 18 supercells


#### Remove an enumeration

- Will raise if the enumeration does not exist

In [24]:
project.enum.remove("supercells_by_volume.3")
project.enum.list()

EnumData:
- id: occ_by_supercell.1
- supercell_set: 13 supercells
- configuration_set: 214 configurations
EnumData:
- id: supercells_by_volume.1
- desc: "Initial supercell enumeration"
- supercell_set: 13 supercells
EnumData:
- id: supercells_by_volume.2
- supercell_set: 15 supercells


## Calculation

There are many different workflow tools for managing DFT calculations that could be interfaced with CASM. The calculation process implemented here involves the following conversions:

    libcasm.configuration.Configuration 
    -> libcasm.xtal.Structure 
    -> ase.Atoms 
    -> VASP input files (POSCAR, KPOINTS, INCAR, POTCAR)

- CASM provides a standard directory structures for saving input files. Using this structure also enables some CASM features for managing jobs on a compute cluster. 
- CASM provides a very simple integration to [The Atomic Simulation Environment (ase)](https://wiki.fysik.dtu.dk/ase/index.html) which can be customized for a particular use case.
- Users can customize the input file setup and output file parsing to work with other DFT codes or more complicated workflows.

### Calculation directory structure

The calculation directories created will be at:

    calc_dir = <enum_dir>/training_data/calctype.<calctype_id>/<configname>/

Where

    enum_dir = <project>/enumerations/enum.<enum_id>/

Here:

- *calctype_id* - The calculation type ID used to organize calculations with different parameter sets or calculating different quantities (i.e. energy vs band structure).
- *configname* - The configuration name used to identify the configuration in the enumeration. Can be obtained from *record.name* when iterating over a [ConfigSelection](TODO).
- *project* - The path to the project directory
- *enum_id* - The enumeration ID


CASM features for running calculations and collecting results use the following format for *\<calc_dir\>*:

- *run.0/* - Directory with initial input files
- *run.final/* - Directory with final calculation results
- *config.json* - The input CASM [Configuration](https://prisms-center.github.io/CASMcode_pydocs/libcasm/configuration/2.0/reference/libcasm/_autosummary/libcasm.configuration.Configuration.html#libcasm.configuration.Configuration)
- *structure.json* - The input CASM [Structure](https://prisms-center.github.io/CASMcode_pydocs/libcasm/xtal/2.0/reference/libcasm/_autosummary/libcasm.xtal.Structure.html#libcasm.xtal.Structure)
- *structure_with_properties.json* - The final calculated CASM [Structure](https://prisms-center.github.io/CASMcode_pydocs/libcasm/xtal/2.0/reference/libcasm/_autosummary/libcasm.xtal.Structure.html#libcasm.xtal.Structure)
- *status.json* - Calculation status
- *submit.sh* - Cluster job submit script.

### Calculation status file

CASM can provide calculation status and help manage Slurm jobs when the *status.json* file includes the following attributes:

- *"status"* - *str*, Calculation status. Standard values are:
  - *"none"* - Calculation not setup or status unknown.
  - *"setup"* - Calculation input files are setup and ready for calculation.
  - *"submitted"* - Calculation job submitted.
  - *"started"* - Job has begun running.
  - *"canceled"* - Job has been canceled, before or after starting.
  - *"stopped"* - Job has been stopped, after it began running, and is not complete. May be an error, hit walltime, a canceled job has stopped, or other reason.
  - *"complete"* - Job has completed successfully.
- *"jobid"* - *str*, The submitted job ID.
- *"starttime"* - *str*, The time the job started, in ISO 8601 format (YYYY-MM-DDTHH:mm:ss)
- *"stoptime"* - *str*, The time the job was stopped, successfully or not, in ISO 8601 format.

### Calculation settings

Calculation settings and template input files are stored in the calculation settings directory:

    calc_settings_dir = <project>/calculation_settings/calctype.<calctype_id>/

This allows easily working with calculations performed for the same configuration with:

- different parameter sets, or
- using different DFT software, or
- calculating different quantities (i.e. energy at fixed volume vs energy after relaxation vs band structure).


#### Calculation settings for ASE + VASP

CASM provides [AseVaspTool](TODO), a simple integration with VASP which makes use of the [ase.calculators.vasp.Vasp](https://wiki.fysik.dtu.dk/ase/ase/calculators/vasp.html) calculator to setup VASP calculations and collect results. It can be used as a template to integrate with other software packages and run more complex workflows.

Here we add input files used by AseVaspTool to the calculation settings directory:

- *INCAR* - A template VASP INCAR file.
- *KPOINTS* - The VASP KPOINTS file
- *calc.json* - Parameters (i.e. choice of pseudopotential) given to the [ase.calculators.vasp.Vasp](https://wiki.fysik.dtu.dk/ase/ase/calculators/vasp.html) constructor
- *meta.json* - Description of the calculation type and any other metadata
- *vasp_relax_greatlakes.sh* - Template Slurm submit script used for performing a series of  VASP structure relaxation runs (ionic positions, cell volume, and cell shape) followed by a static energy calculation. This example is for the [Great Lakes HPC Cluster](https://its.umich.edu/advanced-research-computing/high-performance-computing/great-lakes).

In [25]:
import os

# Give your calculation type a name
calctype_id = "vasp-parameter-set-1"

# Create a new clex (CLuster EXpansion) description for
# a cluster expansion that uses this calctype, and set it as the default.
# Setting it as the default makes this calculation type 
# the default calculation type used by ConfigSelection. 
project.settings.add_clex(name=calctype_id, calctype=calctype_id)
project.settings.set_default_clex(name=calctype_id)
project.commit_settings()

# A helper for setting up and collecting calculation files:
calc = project.calc.get(calctype_id)

# Store a description of the calculation type
# The `desc` attribute is printed by `project.calc.list()`
calc.meta = dict(
    desc=(
        "VASP relaxation energy. "
        "PBE(Si,Ge_d). Rk=35. ENCUT=400. EDIFF=1e-05. EDIFFG=-2e-02."
    ),
)

# Save changes to the `meta` attribute
calc.commit()

### Write template INCAR file
#
# Used in calculation setup as input to ase.calculators.vasp.Vasp.read_incar
#
incar_text = """\
ENCUT = 400 #cutoff
ISPIN = 1 #does non spin-polarized calc.
ALGO = Fast
PREC = Accurate #cutoff + wrap around errors.
IBRION = 2 #conj. grad. relaxation.
NSW = 61 #numberof ionic steps taken in minimization. Make it odd.
ISIF = 3 #whether stress tensor is calculated, what is allowed to relax.
ISMEAR = 1 #BZ integration method (for relaxation runs).
SIGMA = 0.2 #smearing width (keep T*S < 1meV/atom). 
EDIFF = 1e-05 #electronic convergence
EDIFFG = -2e-02 #ion convergence
LWAVE = .FALSE.
LCHARG = .FALSE.
NCORE = 6 #number cores per orbital
"""
calc.write_text_file(
    name="INCAR",
    text=incar_text,
)

### Write template KPOINTS file
#
# Used in calculation setup as input to ase.calculators.vasp.Vasp.read_kpoints
#
kpoints_text = """\
Fully automatic mesh
0              ! 0 -> automatic generation scheme 
Auto           ! fully automatic
  35           ! length (R_k)
"""
calc.write_text_file(
    name="KPOINTS",
    text=kpoints_text,
)

### Write "calc.json"
#
# Constructor arguments for ase.calculators.vasp.Vasp
# 
calc.write_json_file(
    name="calc.json",
    data=dict(
        setups=dict(Si="", Ge="_d"),
        xc="pbe",
    ),
)

### Copy template submit script
#
# Used to generate submit.sh scripts for each calculation
# 
calc.add_file(input_dir / "scripts/vasp_relax_greatlakes.sh")

# List calculation types:
project.calc.list()
print()

# List calculation settings files:
calc.list()
print()

overwrite: SiGe_occ/.casm/project_settings.json
overwrite: SiGe_occ/calculation_settings/calctype.vasp-parameter-set-1/meta.json
overwrite: SiGe_occ/calculation_settings/calctype.vasp-parameter-set-1/INCAR
overwrite: SiGe_occ/calculation_settings/calctype.vasp-parameter-set-1/KPOINTS
overwrite: SiGe_occ/calculation_settings/calctype.vasp-parameter-set-1/calc.json
CalcData:
- id: default
CalcData:
- id: vasp-parameter-set-1
- desc: "VASP relaxation energy. PBE(Si,Ge_d). Rk=35. ENCUT=400. EDIFF=1e-05. EDIFFG=-2e-02."

'vasp-parameter-set-1' settings files:
- INCAR
- vasp_relax_greatlakes.sh
- calc.json
- KPOINTS
- meta.json



### Setup calculations

#### Select configurations

In [26]:
# Enumeration, calculation type, and selection to use
enum_id = "occ_by_supercell.1"
clex_id = "vasp-parameter-set-1"
selection_name = "main"

# Create `enum` and `calc` objects
project.settings.set_default_clex(name=clex_id)
clex_desc = project.settings.get_clex(clex_id)
enum = project.enum.get(enum_id)
calc = project.calc.get(clex_desc.calctype)

# Create selection
config_selection = enum.config_selection(selection_name)

# --- Select configurations ---

# Select configurations based on lambda value
# Start with a few configurations to test setup before trying
# to set up all calculations in the enumeration
config_selection.set_selected(lambda record: record.n_unitcells == 1)

# # Select all configurations
# config_selection.select_all()

# Save selection
config_selection.commit()

# Summary
print(config_selection)

write: SiGe_occ/enumerations/enum.occ_by_supercell.1/config_selection.main.json
ConfigSelection:
- name: main
- enum: occ_by_supercell.1
- clex: vasp-parameter-set-1
  - calctype: vasp-parameter-set-1
  - ref: default
  - bset: default
  - eci: default
- n_total: 214
- n_selected: 3
- n_unselected: 211


#### Generate VASP input files

The [calc.setup](TODO) method iterates over configurations in the ConfigSelection and runs [AseVaspTool.setup](TODO) to create VASP input files for selected configurations. 

To avoid overwriting existing results, [calc.setup](TODO) skips configurations with a calculation status other than "none".

In [27]:
import os

# !! CHANGE THIS AS NECESSARY !!
# write ASE config.ini file, set
# VASP_PP_PATH for ase.calculators.vasp.Vasp
vasp_pp_path = input_dir / "dummy_vasp_potentials/"
config_ini_path = input_dir / "config.ini"
config_ini_path.write_text(
    f"[environment]\nVASP_PP_PATH = {str(vasp_pp_path.resolve())}\n"
)
os.environ["ASE_CONFIG_PATH"] = str(config_ini_path.resolve())

# Use existing `calc` from previous cell

# Setup calculations using the chosen calculation type
calc.setup(
    config_selection=config_selection,
    tool="vasp",
)

skipping: SCEL1_1_1_1_0_0_0/0 (status=setup)
skipping: SCEL1_1_1_1_0_0_0/1 (status=setup)
skipping: SCEL1_1_1_1_0_0_0/2 (status=setup)




#### Check input files

Input files can be checked by inspecting them through the file browser.

Here we check input files are written by iterating over the ConfigSelection:

In [28]:
# Check selected calculations
for record in config_selection:
    print("---")
    print(f"{record.name}:")
    print(record.calc_dir)
    !ls -hl {record.calc_dir}
    !ls -hl {record.calc_dir / "run.0"}
    print()

    # Did you find an error? Remove setup files with:
    # !rm -r {record.calc_dir}


---
SCEL1_1_1_1_0_0_0/0:
/Users/bpuchala/codes/CASM_v2_source/CASMcode_modules/CASMcode_project/notebooks/SiGe_occ/enumerations/enum.occ_by_supercell.1/training_data/calctype.vasp-parameter-set-1/SCEL1_1_1_1_0_0_0/0
total 32
-rw-r--r--  1 bpuchala  staff   192B Feb 28 11:12 config.json
drwxr-xr-x  7 bpuchala  staff   224B Feb 28 11:12 run.0
-rw-r--r--  1 bpuchala  staff    24B Feb 28 11:12 status.json
-rw-r--r--  1 bpuchala  staff   255B Feb 28 11:12 structure.json
-rw-r--r--  1 bpuchala  staff   3.3K Feb 28 11:24 submit.sh
total 40
-rw-r--r--  1 bpuchala  staff    26B Feb 28 11:12 ase-sort.dat
-rw-r--r--  1 bpuchala  staff   202B Feb 28 11:12 INCAR
-rw-r--r--  1 bpuchala  staff    65B Feb 28 11:12 KPOINTS
-rw-r--r--  1 bpuchala  staff   369B Feb 28 11:12 POSCAR
-rw-r--r--  1 bpuchala  staff     8B Feb 28 11:12 POTCAR

---
SCEL1_1_1_1_0_0_0/1:
/Users/bpuchala/codes/CASM_v2_source/CASMcode_modules/CASMcode_project/notebooks/SiGe_occ/enumerations/enum.occ_by_supercell.1/training_data/cal

#### Checking calculation status

On the command line, the *casm-calc status* tool can be used to check calculation status from anywhere inside the project directory:

In [29]:
# To see options:
# !casm-calc status -h

# usage: casm-calc status [-h] [-c SELECTION] [-a] [--calctype CALCTYPE]
#                         [--clex CLEX] [-t] [-d] [--none] [--setup] [--started]
#                         [--submitted] [--canceled] [--stopped] [--complete]
#                         [--other] [-l]
#                         [enum]
# 
# Check the status of CASM project calculations

!cd {project.path} && casm-calc status -c main occ_by_supercell.1 -td

ConfigSelection:
- name: main
- enum: occ_by_supercell.1
- clex: vasp-parameter-set-1
  - calctype: vasp-parameter-set-1
  - ref: default
  - bset: default
  - eci: default
- n_total: 214
- n_selected: 3
- n_unselected: 211

Name                                Status      Job ID      Runtime           
------------------------------------------------------------------------------
SCEL1_1_1_1_0_0_0/0                 setup       none        none              
SCEL1_1_1_1_0_0_0/1                 setup       none        none              
SCEL1_1_1_1_0_0_0/2                 setup       none        none              

Status      Count
--------  -------
setup           3



#### Generate Slurm job submission scripts

Here we generate the Slurm job submission scripts (*submit.sh*) by:

- Looping over selected configurations,
- calculating the job submission parameters based on the number of atoms, and
- rendering a template script using the Jinja templating engine.


In [30]:
import math

from casm.tools.shared.text_io import safe_write
from jinja2 import Environment, FileSystemLoader

# Choose submit script template to use
script_template = "vasp_relax_greatlakes.sh"

# Use existing `config_selection` from previous cells
# Use existing `calc` from previous cells

# Jinja2 submit script rendering setup
env = Environment(
    loader=FileSystemLoader(calc.settings_dir),
    autoescape=False,  # Set to False for shell scripts
)

# Slurm / VASP typical params on greatlakes:
# !! CHANGE THIS AS NECESSARY !!
# - INCAR tag ncore = ~6
# - ~ 1 atom / task
# - 1 core / task
# - 36 cores / node in standard partition
# - ntasks: set to smallest multiple of ncore >= n_atoms / atoms_per_task
# - nodes: set to int(ceil(ntasks / cores_per_node))
ncore = 6
atoms_per_task = 1
cores_per_node = 36

# Loop over selected configurations
for record in config_selection:
    
    # --- Set script template variables ---
    # Number of atoms
    n_atoms = len(record.structure.atom_type())

    # Number of tasks to request
    ntasks_needed = int(math.ceil(n_atoms / atoms_per_task))
    ntasks = 0
    while ntasks < ntasks_needed:
        ntasks += ncore

    # Number of nodes
    nodes = int(math.ceil(ntasks / cores_per_node))

    # Script template variables
    template_vars = dict(
        job_name=f'"{record.name}"',
        time="48:00:00",
        nodes=nodes,
        ntasks=ntasks,
        mem_per_cpu="3800m",
        account="prisms_project1",
        partition="standard",
        vasp_version="5.4.4.18Apr17.p1",
        imax=4,  # Max number of vasp relaxation runs
    )

    # --- Render script ---
    text = env.get_template(script_template).render(template_vars)
    safe_write(
        text=text,
        path=record.calc_dir / "submit.sh",
        force=True,
    )


overwrite: SiGe_occ/enumerations/enum.occ_by_supercell.1/training_data/calctype.vasp-parameter-set-1/SCEL1_1_1_1_0_0_0/0/submit.sh
overwrite: SiGe_occ/enumerations/enum.occ_by_supercell.1/training_data/calctype.vasp-parameter-set-1/SCEL1_1_1_1_0_0_0/1/submit.sh
overwrite: SiGe_occ/enumerations/enum.occ_by_supercell.1/training_data/calctype.vasp-parameter-set-1/SCEL1_1_1_1_0_0_0/2/submit.sh


### Run calculations

Typically, calculations are performed by submitting jobs on a HPC cluster from the login node. If this notebook is running through OnDemand on a compute node it cannot submit jobs directly or they may be submitted but not run successfully. To get around this, we will:

- Select configurations to submit,
- ssh to the HPC cluster login node, and
- on the command line, use *casm-calc submit* to submit jobs.


#### Select configurations

In [31]:
# Enumeration, calculation type, and selection to use
enum_id = "occ_by_supercell.1"
clex_id = "vasp-parameter-set-1"
selection_name = "main"

project.settings.set_default_clex(name=clex_id)
clex_desc = project.settings.get_clex(clex_id)
enum = project.enum.get(enum_id)

# Create selection
config_selection = enum.config_selection(selection_name)
config_selection.set_selected(lambda record: record.n_unitcells == 1)
config_selection.commit()
print(config_selection)

overwrite: SiGe_occ/enumerations/enum.occ_by_supercell.1/config_selection.main.json
ConfigSelection:
- name: main
- enum: occ_by_supercell.1
- clex: vasp-parameter-set-1
  - calctype: vasp-parameter-set-1
  - ref: default
  - bset: default
  - eci: default
- n_total: 214
- n_selected: 3
- n_unselected: 211


#### Submit jobs: casm-calc submit

From the command line on the login node, use the *casm-calc submit* tool from anywhere within the project directory to actually submit the jobs.

Additional notes on *casm-calc submit*:

- When it submits jobs, the jobid is saved in the *status.json* file and the calculation status is set to "submitted".
- Only calculations with status="setup" and jobid="none" will be submitted.
- If there is a problem, the *--cancel* option allows canceling submitted jobs.

<div style="background-color: #ffebb9; padding: 10px; margin: 10px;">
    <b style="color: #B27000;">Note:</b> The <em>vasp_relax_greatlakes.sh</em> script supports continuing relaxation calculations, starting from the last <em>run.$I</em> directory. Continuing calculations with <em>casm-calc submit</em> requires first setting status="setup" and jobid="none" in <em>status.json</em>. Calculations can also be continued by modifying the input files and using <em>sbatch submit.sh</em> from the login node.
</div>

Here we use *casm-calc submit* with the *--dry-run* option to test which jobs will be submitted without actually submitting them:

In [32]:
# To see options:
# !casm-calc submit -h

# usage: casm-calc submit [-h] [-c SELECTION] [--calctype CALCTYPE]
#                         [--clex CLEX] [--dry-run] [--cancel] [-l]
#                         [enum]

# Submit CASM project calculations

!cd {project.path} && casm-calc submit -c main occ_by_supercell.1 --dry-run

ConfigSelection:
- name: main
- enum: occ_by_supercell.1
- clex: vasp-parameter-set-1
  - calctype: vasp-parameter-set-1
  - ref: default
  - bset: default
  - eci: default
- n_total: 214
- n_selected: 3
- n_unselected: 211

Name                                Status      Job ID      Message           
------------------------------------------------------------------------------
SCEL1_1_1_1_0_0_0/0                 setup       none        (dry-run)         
SCEL1_1_1_1_0_0_0/1                 setup       none        (dry-run)         
SCEL1_1_1_1_0_0_0/2                 setup       none        (dry-run)         


#### Check job status

**Using *squeue*:** 

Job status can be checked using `squeue`:

In [33]:
!squeue -u $USER

zsh:1: command not found: squeue


**Using ConfigSelection.tabulate_calc_status:**

Job status can also be checked using ConfigSelection.tabulate_calc_status:

In [34]:
status_selection = enum.config_selection("main")
status_selection.tabulate_calc_status()

Status       Count
---------  -------
none             0
setup            3
submitted        0
started          0
canceled         0
stopped          0
complete         0


**Using ConfigSelectionRecord:**

Job status can also be checked by iterating over a ConfigSelection and checking *calc_status*, *calc_jobid*, and *calc_runtime*.

In [35]:
# A nice header
print(f"{'Name':36}{'Status':12}{'Job ID':12}{'Runtime':18}")
print("-" * 78)

# Check selected calculations:
for record in status_selection:
    name = record.name
    status = record.calc_status
    jobid = record.calc_jobid
    runtime = record.calc_runtime
    print(f"{name:36}{status:12}{jobid:12}{runtime:18}")


Name                                Status      Job ID      Runtime           
------------------------------------------------------------------------------
SCEL1_1_1_1_0_0_0/0                 setup       none        none              
SCEL1_1_1_1_0_0_0/1                 setup       none        none              
SCEL1_1_1_1_0_0_0/2                 setup       none        none              


## Import and Mapping

### Overview

The import and mapping process is essentially the reverse of the calculation setup process:

    VASP output files (vasprun.xml)
    -> ase.Atoms
    -> libcasm.xtal.Structure 
    -> libcasm.configuration.Configuration 

However, while the Configuration -> Structure conversion is deterministic, the relaxed Structure -> Configuration mapping is not. It is common to have orderings that are unstable and experience large lattice changes or atomic displacements during relaxation (see Kolli, Natarajan, and Van der Ven, Acta Materialia 221 (2021) 117429). 

For example, consider:

- a neighboring atom is unstable and relaxes into an originally vacant site, or
- an ordering on one lattice (i.e. HCP) that is unstable and relaxes with large lattice deformation and atomic shuffles, becoming an ordering on another lattice (i.e. BCC).

Before fitting a cluster expansion to our calculation results, we must find the proper configuration for each relaxed structure and check for structures that are either (i) a different configuration or (ii) not any configuration of the current prim.

### Import

#### Import results file

To prepare for mapping, the import process collects for each calculation:

- *config* - *optional*, The initial configuration
- *structure* - *optional*, The initial structure
- *structure_with_properties* - *required*, The final calculated structure

When we know the initial configuration, we can constrain the mapping search to the only consider lattice mappings to the initial supercell. If the initial supercell is not known, then a more exhaustive search must be performed.

Especially for prim in which vacancies are allowed, constraining the lattice mapping can make the overall mapping process faster and more robust. In cases of large relaxation this assumption might result in missing the globally best mapping, but the best mapping found under the constraint will have a large mapping cost and can be safely screened out. Constraining the lattice also has the beneficial feature of eliminating unintuitive mappings to equivalent but reoriented configurations.

The result of an import is a JSON file with format:

```json
{
    <relpath>: {
        "config": <Configuration, optional>,
        "structure": <Structure, optional>,
        "structure_with_properties": <Structure, required>
    },
    ...
}
```

where *\<relpath\>* gives the relative path from the file to the calculation directory, the "structure_with_properties" is required, and the "config" and "structure" are optional. Using relative paths has the advantage that the original calculations and import results can be more easily shared across different system, as long as their relative locations remain fixed.

The import results file can be stored in the parent of the directory being imported. For example:

```text
training_data/
├── calctype.vasp-parameter-set-1/
├── calctype.vasp-parameter-set-1.results.json
...
```

#### Importing VASP calculations

The *import_directory* method walks a directory heirarchy and uses a report handler to (i) check if each subdirectory is a calculation directory, (ii) check if the calculation is completed, and (iii) report the calculation output as a CASM structure with the calculated properties, along with the initial *config* and *structure* if they are available.

**Input**

*CasmVaspReportHandler* is designed for importing VASP calculations from CASM enumerations, but it also works to import DFT calculations setup and run outside of CASM, as long as the the calculations are organized according to the following rules:

1. A directory containing a file named *status.json* is a calculation directory.
2. No calculation directories exist inside another calculation directory.
3. A calculation directory with a *status.json* file containing status="complete" in
   a top-level JSON attribute is complete. Otherwise, the calculation is
   incomplete.
4. Results are reported from the *run.final* directory inside the completed
   calculation directory. By default, *AseVaspTool* is used, which reports the
   results from *OUTCAR* or *OUTCAR.gz*. 


In summary:

```text

target_dir/
├── .../calculation_dir/
│   ├── config.json          # Optional
│   ├── structure.json       # Optional
│   ├── status.json          # Required
│   ├── ...
│   ├── run.final/           # Required
│   │   ├── OUTCAR           # <- One is required
│   │   ├── OUTCAR.gz        # <- 
│   │   ...
│   └── structure_with_properties.json  # Generated
...
```

**Output**

The resulting CASM structures with properties will be written to the calculation directory to speed up subsequent imports, in case more calculations are run.

The *import_directory* method writes three files:

- *\<target_dir\>.results.json*: The import results file, formatted as described in the previous section. If *config.json* and *structure.json* are found in a completed calculation directory they will be included.
- *\<target_dir\>.complete.json*: A list of calculations directories that were found, and were complete.
- *\<target_dir\>.incomplete.json*: A list of calculations directories that were found, but were not complete.

The current version of *import_directory* writes *structure_with_properties.json* to the completed calculations directory, and re-uses that result if it exists. Future versions will instead check the *\<target_dir\>.results.json* file if it already exists.


#### Import using Python

Here we demonstrate selecting a calculation directory from an enumeration and using *import_directory*:

In [36]:
from casm.tools.calc.methods import import_directory
from casm.tools.calc.handlers import CasmVaspReportHandler
from casm.tools.shared.json_io import read_required

enum_id = "occ_by_supercell.1"
calctype_id = "vasp-parameter-set-1"
update = False # If True, force update by re-parsing OUTCAR

target_dir = project.enum.get(enum_id).calctype_dir(calctype_id)

if not target_dir.exists():
    print("Target directory does not exist")
    print(f"target_dir={target_dir}")
else:
    # Import calculations:
    # - Import all VASP calculations in <target_dir>:
    #   - Found by <calc_dir>/status.json file with status="complete"
    #   - Parse <calc_dir>/run.final/OUTCAR
    # - Write results to: <target_dir>.results.json
    import_directory(
        dir=target_dir,
        handler=CasmVaspReportHandler(update=update),
    )
   

Looking for calculations in /Users/bpuchala/codes/CASM_v2_source/CASMcode_modules/CASMcode_project/notebooks/SiGe_occ/enumerations/enum.occ_by_supercell.1/training_data/calctype.vasp-parameter-set-1 ...
#Complete: 0, #Incomplete: 0
#Complete: 0, #Incomplete: 3
#Complete: 0, #Incomplete: 3
overwrite: SiGe_occ/enumerations/enum.occ_by_supercell.1/training_data/calctype.vasp-parameter-set-1.complete.json
overwrite: SiGe_occ/enumerations/enum.occ_by_supercell.1/training_data/calctype.vasp-parameter-set-1.incomplete.json
overwrite: SiGe_occ/enumerations/enum.occ_by_supercell.1/training_data/calctype.vasp-parameter-set-1.results.json


#### Import using CLI

The *import_directory* method can also be used from the command line with:

In [37]:
# Import via CLI tool
!casm-calc vasp import {target_dir}

Looking for calculations in /Users/bpuchala/codes/CASM_v2_source/CASMcode_modules/CASMcode_project/notebooks/SiGe_occ/enumerations/enum.occ_by_supercell.1/training_data/calctype.vasp-parameter-set-1 ...
#Complete: 0, #Incomplete: 0
#Complete: 0, #Incomplete: 3
#Complete: 0, #Incomplete: 3
overwrite: SiGe_occ/enumerations/enum.occ_by_supercell.1/training_data/calctype.vasp-parameter-set-1.complete.json
overwrite: SiGe_occ/enumerations/enum.occ_by_supercell.1/training_data/calctype.vasp-parameter-set-1.incomplete.json
overwrite: SiGe_occ/enumerations/enum.occ_by_supercell.1/training_data/calctype.vasp-parameter-set-1.results.json


#### Imports from other software programs

To import calculations performed using packages other than VASP, options include:

- Copy and adapt *AseVaspTool* to read calculation results using other ASE calculators.
- Copy and adapt *CasmVaspReportHandler* to find completed calculation directories if the CASM layout with *status.json* files and *run.final* directories is not easily used.
- A custom parser can construct CASM Structure and build the import results file directly.

In [38]:
!ls {target_dir}

SCEL1_1_1_1_0_0_0


In [39]:

# Read results
results_path = pathlib.Path(str(target_dir) + ".results.json")
results = read_required(results_path)
 
# Print the first 3 results
# - For standard enumeration layout relpath=configuration_name
print("\n\nFirst 3 results:")
for i, relpath in enumerate(results):
    if i == 3:
        break
    print("---")
    print(f"RelativePath: {relpath}")
    print(xtal.pretty_json(results[relpath]))
print()



First 3 results:



### Mapping

#### The mapping process

In general, there are two main steps for mapping imported structures:

1. For each calculated structure, find the best mappings to ideal configurations
2. For each configuration in an enumeration, find calculated structures that were mapped to it or a symmetrically equivalent configuration. If there are multiple possible mappings, select one using a minimum energy or minimum deformation rule.

In summary:

    Calculated structure w/ properties
    -> Mapped structure w/ properties
    -> Mapped configuration w/ properties
    -> Canonical configuration w/ properties
    
    Enumerated configuration
    -> Canonical configuration
    -> Search for equivalent canonical configurations w/ properties
    -> Conflict resolution 
    -> Mapped configuration w/ properties

Notes:

- It is possible that a calculated fails to map because it has relaxed with such large deformation that it doesn't map to any ideal configuration of the prim. A maximum total mapping cost determines if any mapping should be accepted.
- In this tutorial, it is known that the all the calculated structures map back to the initial configuration and the search and conflict resolution process is skipped.


#### Mapping results file



#### Unzip precalculated data

For the Si-Ge tutorial:

- pre-calculated and imported VASP results are provided
- includes Si-Ge orderings in supercells up to size 4

Here we unzip the data:

In [40]:
import zipfile

from casm.tools.shared.json_io import read_required

enum_id = "occ_by_supercell.1"
calctype_id = "vasp-parameter-set-1"

# The pre-calculated data will be put where
# it would have been imported.
target_dir = project.enum.get(enum_id).calctype_dir(calctype_id)
extract_dir = target_dir.parent

# Unzip precalculated imported calculations
zip_path = input_dir / "SiGe_occ" / "v2" / "precalculated.zip"
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)


#### Map imported structures

In the following we:

- Read the imported structures
- Use [map_structures]() to search for structure mappings
- Store the best mapping found in a dict, using *\<relpath\>* as key
- Store lists of structures that mapped successful and structures that failed to map

For the Si-Ge prim, there are no vacancies so we can restrict the mapping search to only supercells of a particular volume determined from the number of atoms.

In [41]:
import libcasm.mapping.methods as mapmethods

# Read import results
import_results_path = pathlib.Path(str(target_dir) + ".results.json")
import_results = read_required(import_results_path)

# Collect results here:
failed_structures = []
mapped_structures = []
mapping_results = {}

for relpath, data in import_results.items():
    
    # Structure to map
    structure_with_properties = xtal.Structure.from_dict(
        data=data.get("structure_with_properties"),
    )
    structure_with_properties_fg = xtal.make_structure_factor_group(
        structure_with_properties
    )

    # no vacancies, exact volume
    xtal_prim = project.prim.xtal_prim
    vol = int(len(structure_with_properties.atom_type()) / len(xtal_prim.occ_dof()))

    # find mappings
    mappings = mapmethods.map_structures(
        prim=xtal_prim,
        structure=structure_with_properties,
        max_vol=vol,
        prim_factor_group=project.prim.factor_group.elements,
        structure_factor_group=structure_with_properties_fg,
        min_vol=vol,
        min_cost=0.0,
        max_cost=1e20,
        lattice_cost_weight=0.5,
        lattice_cost_method="isotropic_strain_cost",
        atom_cost_method="isotropic_disp_cost",
        k_best=1,
    )

    mapping_data = {}

    # Check mapping results:
    if len(mappings) == 0:
        failed_structures.append(relpath)
    else:
        mapping_data["scored_structure_mapping"] = mappings[0]
        mapped_structures.append(relpath)

    mapping_results[relpath] = mapping_data

print("Finished mapping structures:")
print(f"- # Successful mappings: {len(mapped_structures)}")
print(f"- # Failed mappings: {len(failed_structures)}")

Finished mapping structures:
- # Successful mappings: 214
- # Failed mappings: 0


### Map structures to configurations

- Here we just find and keep the best mapping.
- In some cases, additional next-best mappings may be kept.

#### Construct mapped configurations with properties

In [42]:
from libcasm.configuration import (
    ConfigurationSet,
    ConfigurationWithProperties,
    make_canonical_configuration,
)

enum = project.enum.get(enum_id)

# Make mapped structures,
# from calculated structure and structure mapping,
# then make ConfigurationWithProperties
configuration_set = ConfigurationSet()
for relpath, mapping_data in mapping_results.items():
    print(f"---")
    print(f"relpath={relpath}")

    import_data = import_results[relpath]

    structure_with_properties = xtal.Structure.from_dict(
        data=import_data.get("structure_with_properties"),
    )

    # Apply structure mapping to make mapped structure
    mapped_structure = mapmethods.make_mapped_structure(
        structure_mapping=mapping_data.get("scored_structure_mapping"),
        unmapped_structure=structure_with_properties,
    )
    mapping_data["mapped_structure"] = mapped_structure

    # Convert mapped structure to a ConfigurationWithProperties
    mapped_config_with_props = ConfigurationWithProperties.from_structure(
        prim=project.prim,
        structure=mapped_structure,
        supercells=enum.supercell_set,
    )
    print("Mapped configuration with properties:", mapped_config_with_props)
    mapped_canonical_config = make_canonical_configuration(
        mapped_config_with_props.configuration,
        in_canonical_supercell=True,
    )
    energy = mapped_config_with_props.scalar_global_property_value("energy")
    mapping_data["mapped_configuration_with_properties"] = mapped_config_with_props

    record = configuration_set.add(mapped_canonical_config)
    mapping_data["canonical_configuration_name"] = record.configuration_name
    print(f"canonical_form={record.configuration_name}, energy={energy}")
    print()

print("Finished making mapped configuration:")
print(f"- # of mapped structures: {len(mapped_structures)}")
print(f"- # of mapped configurations: {len(configuration_set)}")

---
relpath=SCEL1_1_1_1_0_0_0/0
Mapped configuration with properties: {
  "configuration": {
    "basis": "standard",
    "dof": {
      "occ": [0, 0]
    },
    "supercell_name": "SCEL1_1_1_1_0_0_0",
    "transformation_matrix_to_supercell": [
      [-1, -1, -1],
      [0, 0, 1],
      [0, 1, 0]
    ]
  },
  "global_properties": {
    "Ustrain": {
      "values": [0.9765292503571437, 0.9765292503571439, 0.9765292503571439, 2.6624426372753403e-17, 0.0, 0.0]
    },
    "energy": {
      "values": [-10.84939425]
    }
  },
  "local_properties": {
    "disp": {
      "values": [
        [4.86672570631573e-07, -4.866725713537079e-07, -4.866725713537079e-07],
        [-4.86672570631573e-07, 4.866725713537079e-07, 4.866725713537079e-07]
      ]
    },
    "force": {
      "values": [
        [0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0]
      ]
    }
  }
}
canonical_form=SCEL1_1_1_1_0_0_0/0, energy=-10.84939425

---
relpath=SCEL1_1_1_1_0_0_0/1
Mapped configuration with properties: {
  "configurat

In [43]:
# Just a useful function for plot formatting #
from bokeh.io import output_notebook

output_notebook()


def format_plot(p):
    # p.xaxis.axis_label = x_label
    # p.yaxis.axis_label = y_label

    font_size_1 = "14pt"
    font_size_2 = "10pt"
    font_name = "helvetica"

    p.title.text_font = font_name
    p.title.text_font_size = font_size_1

    p.xaxis.axis_label_text_font = font_name
    p.xaxis.axis_label_text_font_size = font_size_1
    p.xaxis.major_label_text_font = font_name
    p.xaxis.major_label_text_font_size = font_size_2

    p.yaxis.axis_label_text_font = font_name
    p.yaxis.axis_label_text_font_size = font_size_1
    p.yaxis.major_label_text_font = font_name
    p.yaxis.major_label_text_font_size = font_size_2

Loading BokehJS ...

#### Available mapping methods

When performing mappings, the term "parent structure" is used for ideal superstructures of the prim (the Configuration being mapped to), and "child structure" is used for the (possibly relaxed) structure being mapped.

CASM provides several methods to systematically check and score possible mappings:

- [map_structures](https://prisms-center.github.io/CASMcode_pydocs/libcasm/mapping/2.0/reference/libcasm/_autosummary/libcasm.mapping.methods.map_structures.html#map-structures):
  - A very general structure mapping method.
  - Proposes and checks [StructureMapping](https://prisms-center.github.io/CASMcode_pydocs/libcasm/mapping/2.0/reference/libcasm/_autosummary/libcasm.mapping.info.StructureMapping.html#structuremapping) of parent structures to a child structure for a range of parent supercell volumes and lattice vector reorientations, allowing combinations of rigid rotation, translation, lattice strain, and atom displacement.
  - Roughly, the approach is to first propose and check lattice mappings, and then propose and check atom mappings.
  - Mappings are scored using a weighted sum of lattice strain cost and atomic displacement cost metrics.
- [map_lattices](https://prisms-center.github.io/CASMcode_pydocs/libcasm/mapping/2.0/reference/libcasm/_autosummary/libcasm.mapping.methods.map_lattices.html#map-lattices):
  - A lattice mapping method for when the parent superstructure lattice is not known.
  - Proposes and checks [LatticeMapping](https://prisms-center.github.io/CASMcode_pydocs/libcasm/mapping/2.0/reference/libcasm/_autosummary/libcasm.mapping.info.LatticeMapping.html#latticemapping) from parent superstructures to the child structure considering reorientations of the parent lattice vectors.
  - Scores mappings using a lattice strain cost metric.
  - This method is often used inside a loop over parent superstructures.
- [map_lattices_without_reorientation](https://prisms-center.github.io/CASMcode_pydocs/libcasm/mapping/2.0/reference/libcasm/_autosummary/libcasm.mapping.methods.map_lattices_without_reorientation.html#map-lattices-without-reorientation):
  - A lattice mapping method for when the parent superstructure lattice is known.
  - Constructs a [LatticeMapping](https://prisms-center.github.io/CASMcode_pydocs/libcasm/mapping/2.0/reference/libcasm/_autosummary/libcasm.mapping.info.LatticeMapping.html#latticemapping) from the parent superstructure lattice to the child lattice by calculating the lattice deformation gradient directly, without any reorientation of the lattice vectors.
  - Does not score the mapping, but [isotropic_strain_cost](https://prisms-center.github.io/CASMcode_pydocs/libcasm/mapping/2.0/reference/libcasm/_autosummary/libcasm.mapping.info.isotropic_strain_cost.html#isotropic-strain-cost) and [symmetry_breaking_strain_cost](https://prisms-center.github.io/CASMcode_pydocs/libcasm/mapping/2.0/reference/libcasm/_autosummary/libcasm.mapping.info.symmetry_breaking_strain_cost.html#symmetry-breaking-strain-cost) can be used independently.
  - This method is useful when CASM enumerated the initial configuration used for the calculation input.
- [map_atoms](https://prisms-center.github.io/CASMcode_pydocs/libcasm/mapping/2.0/reference/libcasm/_autosummary/libcasm.mapping.methods.map_atoms.html#map-atoms):
  - An atom mapping method for when the lattice mapping is known.
  - Proposes and scores [AtomMapping](https://prisms-center.github.io/CASMcode_pydocs/libcasm/mapping/2.0/reference/libcasm/_autosummary/libcasm.mapping.info.AtomMapping.html#atommapping) displacements from sites to atoms, considering the distinct translations of child atoms to parent structure sublattices.
  - Scores mappings using a atomic displacement cost metric.
  - This method can be used in combination with map_lattices or map_lattices_without_reorientation to get a complete [StructureMapping]().
- The [mapsearch subpackage](https://prisms-center.github.io/CASMcode_pydocs/libcasm/mapping/2.0/reference/libcasm/_autosummary/libcasm.mapping.mapsearch.html#module-libcasm.mapping.mapsearch):
  - Enables the construction of a custom mapping search algorithm.
  - A tutorial on using [libcasm.mapping.mapsearch](https://prisms-center.github.io/CASMcode_pydocs/libcasm/mapping/2.0/reference/libcasm/_autosummary/libcasm.mapping.mapsearch.html#module-libcasm.mapping.mapsearch) is coming soon.

The mapping methods are described in the paper [Thomas, Natarajan, and Van der Ven, npj Computational Materials, 7 (2021), 164](https://doi.org/10.1038/s41524-021-00627-0).

#### Plot lattice and strain mapping costs


In [44]:
import os
import numpy as np
from bokeh.models import ColumnDataSource
from bokeh.plotting import figure, show

names = []
for relpath, import_data in import_results.items():
    names.append(relpath)

all_scores = [x["scored_structure_mapping"] for relpath, x in mapping_results.items()]
total_cost = np.array([x.total_cost() for x in all_scores])
lattice_cost = np.array([x.lattice_cost() for x in all_scores])
atom_cost = np.array([x.atom_cost() for x in all_scores])

data = {
    "names": names,
    "total_cost": total_cost,
    "lattice_cost": lattice_cost,
    "atom_cost": atom_cost,
}
tooltips = [
    ("relpath", "@names"),
    ("total_cost", "@total_cost"),
    ("lattice_cost", "@lattice_cost"),
    ("atom_cost", "@atom_cost"),
]
source = ColumnDataSource(data)

p = figure(width=600, height=400, tooltips=tooltips)
p.scatter("lattice_cost", "atom_cost", source=source, size=5, color="navy", alpha=0.5)
format_plot(p)
p.xaxis.axis_label = "Lattice cost"
p.yaxis.axis_label = "Atom cost"
show(p)

#### Save mapped configurations

In [45]:
# Save structure mappings by input path
results_out = {}
for relpath, x in mapping_results.items():
    results_out[relpath] = {
        "mapped_configuration_with_properties": x[
            "mapped_configuration_with_properties"
        ].to_dict(),
        "scored_structure_mapping": x["scored_structure_mapping"].to_dict(),
        "mapped_structure": x["mapped_structure"].to_dict(),
    }

safe_dump(
    data=results_out,
    path=enum.enum_dir / f"mapping_results.{calctype_id}.json",
    force=True,
)

overwrite: SiGe_occ/enumerations/enum.occ_by_supercell.1/mapping_results.vasp-parameter-set-1.json


## Composition axes

### Introduction

In a crystal with a fixed number of sites, the number of species occupying the same sublattice are not indepedent. In general, a crystal occupied by $s$ component species will have $k<s$ independent compositions. CASM converts between compositions expressed as number of species per unit cell and compositions in terms of independent "parametric composition axes" using:

\begin{align}
    \vec{n} &= \vec{n}_0 + \mathbf{Q} \vec{x} \\
    \vec{x} &= \mathbf{R}^{\mathsf{T}} (\vec{n} - \vec{n}_0)
\end{align}


where:

- $\vec{n}$: Vector of shape=($s$,), the number of each component species
  per unit cell (*mol_composition*).
- $\vec{x}$: Vector of shape=($k$,), The composition along each composition axis when referenced to the origin composition (*param_composition*).
- $\vec{n}_0$: Vector of shape=($s$,), The origin in composition space, as the number of each component species per unit cell.
- $Q$: Matrix of shape=($s$, $k$), with columns representing
the change in composition per unit cell going one unit distance along each independent composition axis.
- $R$: Matrix of shape=($s$, $k$), such that $\mathbf{R}^{\mathsf{T}}\mathbf{Q} = \mathbf{Q}^{\mathsf{T}}\mathbf{R} = \mathbf{I}$.

The "parametric composition axes" are the columns of $Q$, $\vec{q}_i$. Due to preservation of the number of sites per unit cell, $\sum_{i} Q_{ij} = 0$. If vacancies are allowed, they are included as a component species.

### Print standard parametric composition axes

When a CASM project is initialized, a set of standard choices for the parametric axes are determined and stored in the [Project.chemical_composition_axes](TODO) attribute. Printing the chemical composition axes results in a table summarizing the possible choices:

In [46]:
# print possible axes
print(project.chemical_composition_axes)

KEY  ORIGIN    a     GENERAL FORMULA
-----  --------  ----  -----------------
    0  Ge(2)     SiGe  Si(a)Ge(2-a)
    1  Si(2)     SiGe  Si(2-a)Ge(a)

Currently selected composition axes: 1

Parametric composition:
  comp(a) = -0.5*(comp_n(Si) - 2)  + 0.5*comp_n(Ge) 

Composition:
  comp_n(Si) = 2 - 1*comp(a) 
  comp_n(Ge) = 1*comp(a) 

Parametric chemical potentials:
  param_chem_pot(a) = -chem_pot(Si) + chem_pot(Ge)


### Select default parametric composition axes

To select a particular choice as the default composition axes, use [set_current_axes]():


In [47]:
# select axes with <key>, unset with None
project.chemical_composition_axes.set_current_axes(1)

# print possible axes and formulas for current choice
print(project.chemical_composition_axes)

# commit current choice
project.chemical_composition_axes.commit()

KEY  ORIGIN    a     GENERAL FORMULA
-----  --------  ----  -----------------
    0  Ge(2)     SiGe  Si(a)Ge(2-a)
    1  Si(2)     SiGe  Si(2-a)Ge(a)

Currently selected composition axes: 1

Parametric composition:
  comp(a) = -0.5*(comp_n(Si) - 2)  + 0.5*comp_n(Ge) 

Composition:
  comp_n(Si) = 2 - 1*comp(a) 
  comp_n(Ge) = 1*comp(a) 

Parametric chemical potentials:
  param_chem_pot(a) = -chem_pot(Si) + chem_pot(Ge)


Notes:

- In CASM v2, by default the parametric composition axes are "normalized" in the sense that a unit distance along that axis corresponds to a change in occupation of one site per unit cell. In CASM v1, the standard composition axes were not normalized.
- The term "endmember" usually refers to the extreme compositions in a solid solution. In the context of CompositionConverter, the term "end member composition" is used to mean the composition one unit distance along a parametric composition axis, $\vec{n}_0 + \vec{q}_i$.
- When printing formulas, the characters "a", "b", "c", etc. are used to represent the parametric compositions, $x_1$, $x_2$, $x_3$, etc.
- When referring to parametric composition axes, the characters "a", "b", "c", etc. are used to represent the parametric composition axes, $\vec{q}_1$, $\vec{q}_2$, $\vec{q}_3$, etc.

## Formation energy

### Get properties

- Energy per unitcell (eV / unitcell)
- Parameteric composition ($a$ in Si$_{2-a}$Ge$_{a}$)
  - Using [ConfigCompositionCalculator](https://prisms-center.github.io/CASMcode_pydocs/casm/project/2.0/reference/casm/_autosummary/casm.project.ConfigCompositionCalculator.html#configcompositioncalculator)

In [48]:
# This is a ConfigCompositionCalculator
calc_comp = project.make_chemical_comp_calculator()

# Store properties here
relpath_list = []
mapped_configurations = []
energy_per_unitcell = []
comp_a = []

index = 0
for relpath, x in mapping_results.items():
    # Get configuration, energy, and param_composition
    record = x["mapped_configuration_with_properties"]
    configuration = record.configuration
    n_unitcells = configuration.supercell.n_unitcells
    energy = record.scalar_global_property_value("energy")
    param_composition = calc_comp.param_composition(configuration)

    # Append
    mapped_configurations.append(configuration)
    energy_per_unitcell.append(energy / n_unitcells)
    comp_a.append(param_composition[0])
    relpath_list.append(relpath)

    index += 1

### Plot energy per unitcell vs composition

- First plot without setting any energy reference

In [49]:
data = {
    "names": names,
    "total_cost": total_cost,
    "lattice_cost": lattice_cost,
    "atom_cost": atom_cost,
    "energy_per_unitcell": energy_per_unitcell,
    "comp_a": comp_a,
}
tooltips = [
    ("name", "@names"),
    ("total_cost", "@total_cost"),
    ("lattice_cost", "@lattice_cost"),
    ("atom_cost", "@atom_cost"),
    ("energy_per_unitcell", "@energy_per_unitcell"),
    ("comp_a", "@comp_a"),
]
source = ColumnDataSource(data)

p = figure(width=600, height=400, tooltips=tooltips)
p.scatter(
    "comp_a", "energy_per_unitcell", source=source, size=5, color="navy", alpha=0.5
)
format_plot(p)
p.xaxis.axis_label = "Parametric composition (a in Si(2-a)Ge(a))"
p.yaxis.axis_label = "Calculated energy per unitcell"
show(p)

### Calculate formation energy

- Get energy and composition of configurations with min / max $a$

In [50]:
from libcasm.composition import FormationEnergyCalculator

# For 1 independent composition axis:
# get reference state - at max composition
i_max = np.argmax(comp_a)
e_max = energy_per_unitcell[i_max]
comp_a_max = comp_a[i_max]

# get reference state - at min composition
i_min = np.argmin(comp_a)
e_min = energy_per_unitcell[i_min]
comp_a_min = comp_a[i_min]

# Formation energy calculator
e_calc = FormationEnergyCalculator(
    composition_ref=np.array(
        [
            [comp_a_min],
            [comp_a_max],
        ]
    ).transpose(),
    energy_ref=np.array([e_min, e_max]),
)

# Calculate formation energies
formation_energy_per_unitcell = []
for i, config in enumerate(mapped_configurations):
    ef = e_calc.formation_energy(
        composition=calc_comp.param_composition(config), energy=energy_per_unitcell[i]
    )
    formation_energy_per_unitcell.append(ef)

### Save fitting data

- Formation energy per unitcell
- Parameteric composition

In [51]:
safe_dump(
    data={
        "formation_energy_per_unitcell": formation_energy_per_unitcell,
        "comp_a": comp_a,
        "relpath": relpath_list,
    },
    path=enum.enum_dir / f"fitting_data.{calctype_id}.json",
    force=True,
)
        

overwrite: SiGe_occ/enumerations/enum.occ_by_supercell.1/fitting_data.vasp-parameter-set-1.json


### Plot formation energy per unitcell vs composition


In [52]:
data = {
    "names": names,
    "total_cost": total_cost,
    "lattice_cost": lattice_cost,
    "atom_cost": atom_cost,
    "energy_per_unitcell": energy_per_unitcell,
    "formation_energy_per_unitcell": formation_energy_per_unitcell,
    "comp_a": comp_a,
}
tooltips = [
    ("name", "@names"),
    ("total_cost", "@total_cost"),
    ("lattice_cost", "@lattice_cost"),
    ("atom_cost", "@atom_cost"),
    ("energy_per_unitcell", "@energy_per_unitcell"),
    ("formation_energy_per_unitcell", "@formation_energy_per_unitcell"),
    ("comp_a", "@comp_a"),
]
source = ColumnDataSource(data)

p = figure(width=600, height=400, tooltips=tooltips)
p.scatter(
    "comp_a",
    "formation_energy_per_unitcell",
    source=source,
    size=5,
    color="navy",
    alpha=0.5,
)
format_plot(p)
p.xaxis.axis_label = "Parametric composition (a in Si(2-a)Ge(a))"
p.yaxis.axis_label = "Formation energy per unitcell"
show(p)